In [33]:
import pandas as pd

annotations_df = pd.read_csv(SCREENED_TXT, header=0, names=["image_name", "label"])

annotations_df['label'].value_counts()

"""
label
Transverse      387
Longitudinal    379
Unknown         183
Name: count, dtype: int64
"""

'\nlabel\nTransverse      387\nLongitudinal    379\nUnknown         183\nName: count, dtype: int64\n'

# TRANSVERSE VS LONGITUDINAL COMPARISON

In [32]:
from collections import defaultdict
from PIL import Image, ImageDraw

In [ ]:
import csv
import os
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import ttach as tta
from PIL import Image
from skimage.measure import label as sklabel
from skimage.measure import regionprops
from skimage.transform import resize
import segmentation_models_pytorch_4TorchLessThan120 as smp


# LOAD DATA CONFIGS 

PROJECT_ROOT = Path(
    "/Users/JanayeCheong/Documents/radiomics_segmentation_models/"
    "TNSCUI2020-Seg-Rank1st"
)

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"
MASK_DIR = PROJECT_ROOT / "train_thyroidXL" / "masks"

WEIGHT_C1 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage1_trained_on_size_256.pkl"
)

WEIGHT_C2 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage2_trained_on_size_512.pkl"
)

OUTPUT_DIR = PROJECT_ROOT / "inference_outputs_long_vs_trans_v1_07"
PREDICTION_DIR = OUTPUT_DIR / "predicted_masks"
OVERLAY_DIR = OUTPUT_DIR / "overlays"
METRICS_CSV = OUTPUT_DIR / "metrics.csv"

C1_SIZE = 256
C2_SIZE = 512

C1_TTA = True
C2_TTA = True
USE_C2 = True

ORIMG = False
C1_THRESHOLD = 0.5
C2_THRESHOLD = 0.5
C2_RESIZE_ORDER = 0

SAVE_OVERLAYS = True
CONTINUE_ON_ERROR = True

SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}



In [67]:
SCREENED_TXT = Path(
    "TNSCUI2020-Seg-Rank1st/train_thyroidXL/screened.txt"
)

PATIENT_COMPARISON_DIR = OUTPUT_DIR / "patient_view_comparisons"
PATIENT_METRICS_CSV = OUTPUT_DIR / "patient_view_metrics.csv"

In [68]:
def load_view_annotations(annotation_path: Path) -> Dict[str, str]:
    """
    Read screened.txt annotations.

    Expected format:
    # 8 digits + underscore + 8 digits + underscore + 1 digit
    
    first 8 digits indicate the patient ID

        ,classification
        00003307_17EE0585_0,Transverse 
        00003723_CCF3845C_2,Longitudinal
        ...

    Returns
    -------
    {
        "00003307_17EE0585_0": "Transverse",
        "00003723_CCF3845C_2": "Longitudinal",
        ...
    }
    """
    if not annotation_path.exists():
        raise FileNotFoundError(
            f"View annotation file not found: {annotation_path}"
        )

    annotations: Dict[str, str] = {}

    with annotation_path.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as f:

        reader = csv.reader(f)

        for row in reader:

            # Skip completely blank rows
            if not row:
                continue

            # Some files may contain blank lines
            if len(row) < 2:
                continue

            image_id = row[0].strip()
            classification = row[1].strip()

            # Skip header:
            # ,classification
            if (
                not image_id
                and classification.lower() == "classification"
            ):
                continue

            if not image_id:
                continue

            if not classification:
                classification = "Unknown"

            annotations[image_id.lower()] = classification

    print(
        f"Loaded view annotations for "
        f"{len(annotations)} images."
    )

    return annotations

In [ ]:
def load_screened_patients(
    screened_path: Path,
) -> Dict[str, List[Dict]]:
    """
    Read screened image annotations and group by patient ID.

    # EXAMPLE of file format for loading screened patients: 00003307_17EE0585_0,Transverse

    Patient ID is defined as the first 8 characters.
    """

    if not screened_path.exists():
        raise FileNotFoundError(
            f"Screened annotation file not found: {screened_path}"
        )

    patients = defaultdict(list)

    with screened_path.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as f:

        reader = csv.reader(f)

        for row in reader:

            if not row or len(row) < 2:
                continue

            image_id = row[0].strip()
            view = row[1].strip()

            # Header is:
            # ,classification
            if not image_id:
                continue

            if not view:
                view = "Unknown"

            patient_id = image_id[:8]

            patients[patient_id].append(
                {
                    "image_id": image_id,
                    "view": view,
                }
            )

    return dict(patients)


def build_image_index(
    image_dir: Path,
) -> Dict[str, Path]:

    index: Dict[str, Path] = {}

    for path in discover_images(image_dir):

        key = path.stem.lower()

        if key in index:
            raise ValueError(
                f"Duplicate image stem '{path.stem}' found:\n"
                f"  {index[key]}\n"
                f"  {path}"
            )

        index[key] = path

    return index



In [ ]:
def save_patient_comparison(
    patient_id: str,
    patient_results: List[Dict],
    output_path: Path,
) -> None:
    """
    Save successful overlays for one patient side-by-side.

    Each panel contains:
        - view
        - image ID
        - IoU
        - DSC
    """

    successful = [
        row
        for row in patient_results
        if (
            row["status"] == "ok"
            and row["overlay_path"]
        )
    ]

    # From patient results, if there are less than 2 successful overlays, we cannot create a comparison image. Return
    if len(successful) < 2:
        return

    panels = []

    target_width = 500
    header_height = 85

    # Put Transverse first, then Longitudinal,
    # then Unknown/anything else.
    view_order = {
        "transverse": 0,
        "longitudinal": 1,
        "unknown": 2,
    }

    successful = sorted(
        successful,
        key=lambda row: (
            view_order.get(
                str(row["view"]).lower(),
                99,
            ),
            row["image_id"],
        ),
    )

    for row in successful:

        overlay_path = Path(
            row["overlay_path"]
        )

        if not overlay_path.exists():
            continue

        with Image.open(
            overlay_path
        ) as img:

            overlay = img.convert("RGB")

        scale = (
            target_width
            / overlay.width
        )

        resized_height = int(
            overlay.height * scale
        )

        overlay = overlay.resize(
            (
                target_width,
                resized_height,
            )
        )

        panel = Image.new(
            "RGB",
            (
                target_width,
                resized_height + header_height,
            ),
            "white",
        )

        panel.paste(
            overlay,
            (0, header_height),
        )

        draw = ImageDraw.Draw(panel)

        text = (
            f"{row['view']}\n"
            f"{row['image_id']}\n"
            f"IoU={row['iou']:.4f} | "
            f"DSC={row['dsc']:.4f}"
        )

        draw.multiline_text(
            (10, 8),
            text,
            fill="black",
            spacing=4,
        )

        panels.append(panel)

    if len(panels) < 2:
        return

    title_height = 45

    max_height = max(
        panel.height
        for panel in panels
    )

    total_width = sum(
        panel.width
        for panel in panels
    )

    combined = Image.new(
        "RGB",
        (
            total_width,
            max_height + title_height,
        ),
        "white",
    )

    draw = ImageDraw.Draw(combined)

    draw.text(
        (10, 12),
        f"Patient {patient_id}",
        fill="black",
    )

    x_position = 0

    for panel in panels:

        combined.paste(
            panel,
            (
                x_position,
                title_height,
            ),
        )

        x_position += panel.width

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    combined.save(output_path)

In [71]:
def save_patient_metrics(
    results: List[Dict],
    output_path: Path,
) -> None:

    grouped = defaultdict(list)

    for row in results:

        if row["status"] != "ok":
            continue

        grouped[
            row["patient_id"]
        ].append(row)

    patient_rows = []

    for patient_id, rows in sorted(
        grouped.items()
    ):

        transverse = [
            row
            for row in rows
            if str(
                row["view"]
            ).lower() == "transverse"
        ]

        longitudinal = [
            row
            for row in rows
            if str(
                row["view"]
            ).lower() == "longitudinal"
        ]

        unknown = [
            row
            for row in rows
            if str(row["view"]).lower()
            not in {
                "transverse",
                "longitudinal",
            }
        ]

        def mean_metric(
            selected_rows,
            metric_name,
        ):

            if not selected_rows:
                return ""

            return float(
                np.mean(
                    [
                        row[metric_name]
                        for row
                        in selected_rows
                    ]
                )
            )

        transverse_iou = mean_metric(
            transverse,
            "iou",
        )

        longitudinal_iou = mean_metric(
            longitudinal,
            "iou",
        )

        transverse_dsc = mean_metric(
            transverse,
            "dsc",
        )

        longitudinal_dsc = mean_metric(
            longitudinal,
            "dsc",
        )

        if (
            transverse_iou != ""
            and longitudinal_iou != ""
        ):

            iou_difference = (
                longitudinal_iou
                - transverse_iou
            )

            dsc_difference = (
                longitudinal_dsc
                - transverse_dsc
            )

        else:

            iou_difference = ""
            dsc_difference = ""

        patient_rows.append(
            {
                "patient_id":
                    patient_id,

                "total_images":
                    len(rows),

                "transverse_images":
                    len(transverse),

                "longitudinal_images":
                    len(longitudinal),

                "unknown_images":
                    len(unknown),

                "transverse_mean_iou":
                    transverse_iou,

                "longitudinal_mean_iou":
                    longitudinal_iou,

                "iou_long_minus_trans":
                    iou_difference,

                "transverse_mean_dsc":
                    transverse_dsc,

                "longitudinal_mean_dsc":
                    longitudinal_dsc,

                "dsc_long_minus_trans":
                    dsc_difference,

                "has_both_views":
                    int(
                        len(transverse) > 0
                        and
                        len(longitudinal) > 0
                    ),
            }
        )

    fieldnames = [
        "patient_id",
        "total_images",
        "transverse_images",
        "longitudinal_images",
        "unknown_images",

        "transverse_mean_iou",
        "longitudinal_mean_iou",
        "iou_long_minus_trans",

        "transverse_mean_dsc",
        "longitudinal_mean_dsc",
        "dsc_long_minus_trans",

        "has_both_views",
    ]

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(
            patient_rows
        )

In [72]:
annotations_df

,image_name,label
0,00003307_17EE0585_0,Transverse
1,00003723_CCF3845C_2,Longitudinal
2,00001962_034B467C_1,Transverse
3,00002300_E60B804B_0,Transverse
4,00003387_76B10567_2,Transverse
...,...,...
944,00000578_AAD6BA89_1,Unknown
945,00001334_CEB80814_1,Transverse
946,00003110_ABBD7CBC_0,Unknown
947,00001342_FFEB8295_0,Unknown


In [ ]:
from collections import defaultdict, Counter
from pathlib import Path

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"

# Find images recursively and allow common extensions
image_files = [
    p for p in IMG_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
]

print(f"Total image files found: {len(image_files)}")

# ------------------------------------------------------------
# Group by FIRST 8 CHARACTERS of filename stem
# ------------------------------------------------------------

patients = defaultdict(list)

for image_path in image_files:
    stem = image_path.stem.strip()

    patient_id = stem[:8]

    patients[patient_id].append(image_path)



# Show all patients with MORE THAN ONE image


repeated_patients = {
    patient_id: paths
    for patient_id, paths in patients.items()
    if len(paths) > 1
}

print(f"Unique patients: {len(patients)}")
print(
    f"Patients with >1 image: "
    f"{len(repeated_patients)}"
)

print("\n" + "=" * 70)
print("PATIENTS WITH MULTIPLE IMAGES")
print("=" * 70)

for patient_id, paths in sorted(
    repeated_patients.items(),
    key=lambda x: len(x[1]),
    reverse=True,
):

    print(
        f"\n{patient_id}: "
        f"{len(paths)} images"
    )

    for path in sorted(paths):
        print(f"    {path.name}")

Total image files found: 9541
Unique patients: 3354
Patients with >1 image: 3180

PATIENTS WITH MULTIPLE IMAGES

00004012: 10 images
    00004012_08B25F9C_9.png
    00004012_54792304_4.png
    00004012_60F3C0C2_1.png
    00004012_6E510AB4_8.png
    00004012_91D440E8_3.png
    00004012_AD725ABC_0.png
    00004012_BD239B56_7.png
    00004012_E1AFF0ED_2.png
    00004012_EBF1AAD9_5.png
    00004012_EC553CF6_6.png

00003730: 8 images
    00003730_13600386_2.png
    00003730_2D63A125_0.png
    00003730_429CBA29_5.png
    00003730_489C1FC9_3.png
    00003730_5A34EAFF_1.png
    00003730_5DA17244_3.png
    00003730_AD047E8B_4.png
    00003730_E247AD47_2.png

00003950: 8 images
    00003950_1B7C16DE_2.png
    00003950_2B4C084A_5.png
    00003950_2BE0C739_0.png
    00003950_330114F4_4.png
    00003950_50985A09_3.png
    00003950_953153A7_7.png
    00003950_A752ADE8_1.png
    00003950_CB339AA2_6.png

00004065: 8 images
    00004065_1B7A9C45_8.png
    00004065_77CA138D_7.png
    00004065_78063259_1

In [ ]:
from collections import defaultdict, Counter
from pathlib import Path
import csv


SCREENED_TXT = PROJECT_ROOT / "train_thyroidXL" / "screened_annotations.txt"


# Group screened images by first 8 characters = patient ID


patients = defaultdict(list)

with SCREENED_TXT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as f:

    reader = csv.reader(f)

    for row in reader:

        if not row or len(row) < 2:
            continue

        image_id = row[0].strip()
        view = row[1].strip()

        # Skip header: ,classification
        if not image_id:
            continue

        patient_id = image_id[:8]

        patients[patient_id].append(
            {
                "image_id": image_id,
                "view": view,
            }
        )


# ------------------------------------------------------------
# Keep only repeat patients
# ------------------------------------------------------------

repeat_patients = {
    patient_id: images
    for patient_id, images in patients.items()
    if len(images) > 1
}


print(f"Total unique patients in screened file: {len(patients)}")
print(
    f"Repeat patients in screened file: "
    f"{len(repeat_patients)}"
)



# Print each repeat patient and their images/views


print("\n" + "=" * 70)
print("REPEAT PATIENTS")
print("=" * 70)

for patient_id, images in sorted(
    repeat_patients.items(),
    key=lambda x: len(x[1]),
    reverse=True,
):

    print(
        f"\n{patient_id}: "
        f"{len(images)} screened images"
    )

    for item in images:
        print(
            f"    {item['image_id']} "
            f"| {item['view']}"
        )

Total unique patients in screened file: 839
Repeat patients in screened file: 103

REPEAT PATIENTS

00003110: 4 screened images
    00003110_170514ED_2 | Transverse
    00003110_E741AC77_5 | Longitudinal
    00003110_15944615_1 | Transverse
    00003110_ABBD7CBC_0 | Unknown

00003712: 3 screened images
    00003712_3B60994C_4 | Longitudinal
    00003712_DAB6E827_5 | Unknown
    00003712_755843D3_2 | Transverse

00003720: 3 screened images
    00003720_9D4FE88F_2 | Unknown
    00003720_0F75D2E5_1 | Transverse
    00003720_25838695_0 | Transverse

00004398: 3 screened images
    00004398_5C8B85B8_2 | Transverse
    00004398_60752241_1 | Transverse
    00004398_3C20BCAF_0 | Longitudinal

00004575: 3 screened images
    00004575_0418709F_4 | Longitudinal
    00004575_F8AB8366_2 | Longitudinal
    00004575_36A0B468_0 | Transverse

00001223: 3 screened images
    00001223_5EA0BD62_0 | Transverse
    00001223_866CF2B9_1 | Longitudinal
    00001223_4C9077FE_2 | Longitudinal

00001962: 2 screen

In [75]:
repeat_patients

{'00001962': [{'image_id': '00001962_034B467C_1', 'view': 'Transverse'},
  {'image_id': '00001962_8FFD0F42_0', 'view': 'Longitudinal'}],
 '00002759': [{'image_id': '00002759_67AB9CA9_3', 'view': 'Longitudinal'},
  {'image_id': '00002759_8CE6E95E_2', 'view': 'Transverse'}],
 '00002564': [{'image_id': '00002564_091C8F17_1', 'view': 'Transverse'},
  {'image_id': '00002564_074B4873_2', 'view': 'Unknown'}],
 '00003648': [{'image_id': '00003648_64088A99_0', 'view': 'Longitudinal'},
  {'image_id': '00003648_EE87CC6D_1', 'view': 'Transverse'}],
 '00003108': [{'image_id': '00003108_DD20CD2D_3', 'view': 'Unknown'},
  {'image_id': '00003108_AE5AD07C_1', 'view': 'Transverse'}],
 '00001071': [{'image_id': '00001071_B2AA850F_0', 'view': 'Transverse'},
  {'image_id': '00001071_78AAA4D7_1', 'view': 'Longitudinal'}],
 '00001798': [{'image_id': '00001798_77CFF927_1', 'view': 'Transverse'},
  {'image_id': '00001798_0129C07F_2', 'view': 'Longitudinal'}],
 '00001151': [{'image_id': '00001151_FC0722AE_1', '

In [76]:
distribution = Counter(
    len(images)
    for images in patients.values()
)

print("\nScreened images per patient:")

for n_images, n_patients in sorted(
    distribution.items()
):
    print(
        f"{n_images} image(s): "
        f"{n_patients} patient(s)"
    )


Screened images per patient:
1 image(s): 736 patient(s)
2 image(s): 97 patient(s)
3 image(s): 5 patient(s)
4 image(s): 1 patient(s)


In [77]:
patients

defaultdict(list,
            {'00003307': [{'image_id': '00003307_17EE0585_0',
               'view': 'Transverse'}],
             '00003723': [{'image_id': '00003723_CCF3845C_2',
               'view': 'Longitudinal'}],
             '00001962': [{'image_id': '00001962_034B467C_1',
               'view': 'Transverse'},
              {'image_id': '00001962_8FFD0F42_0', 'view': 'Longitudinal'}],
             '00002300': [{'image_id': '00002300_E60B804B_0',
               'view': 'Transverse'}],
             '00003387': [{'image_id': '00003387_76B10567_2',
               'view': 'Transverse'}],
             '00001362': [{'image_id': '00001362_5EA700B7_0',
               'view': 'Transverse'}],
             '00002722': [{'image_id': '00002722_0B61FD23_2',
               'view': 'Longitudinal'}],
             '00000353': [{'image_id': '00000353_883F0C38_0',
               'view': 'Longitudinal'}],
             '00003891': [{'image_id': '00003891_D1D14835_4',
               'view': 'Unknown

In [78]:
for patient in patients.items():
    print(f"\nPatient ID: {patient[0]}")
    print(f"\nImages: {patient[1]}")


Patient ID: 00003307

Images: [{'image_id': '00003307_17EE0585_0', 'view': 'Transverse'}]

Patient ID: 00003723

Images: [{'image_id': '00003723_CCF3845C_2', 'view': 'Longitudinal'}]

Patient ID: 00001962

Images: [{'image_id': '00001962_034B467C_1', 'view': 'Transverse'}, {'image_id': '00001962_8FFD0F42_0', 'view': 'Longitudinal'}]

Patient ID: 00002300

Images: [{'image_id': '00002300_E60B804B_0', 'view': 'Transverse'}]

Patient ID: 00003387

Images: [{'image_id': '00003387_76B10567_2', 'view': 'Transverse'}]

Patient ID: 00001362

Images: [{'image_id': '00001362_5EA700B7_0', 'view': 'Transverse'}]

Patient ID: 00002722

Images: [{'image_id': '00002722_0B61FD23_2', 'view': 'Longitudinal'}]

Patient ID: 00000353

Images: [{'image_id': '00000353_883F0C38_0', 'view': 'Longitudinal'}]

Patient ID: 00003891

Images: [{'image_id': '00003891_D1D14835_4', 'view': 'Unknown'}]

Patient ID: 00004035

Images: [{'image_id': '00004035_67EE1254_1', 'view': 'Transverse'}]

Patient ID: 00000689

Ima

In [79]:
def thyroidxl_preprocess(
    image_path: Path,
    outputsize: C1_SIZE,
    remove_black_edges: bool = True,
):
    """
    ThyroidXL-safe replacement for TNSCUI_preprocess. 
    (original function in utils:
        removes irrelevant edges based on a threshold system of averaged pixels row-wise; the default output_size is 256 x 256
        returns the cut image tensor, coordinates of the min and max x and y FROM ORIGINAL IMAGE respectively that the extracted image without irrelevant
        areas contains  

    Returns
    -------
    processed_tensor:
        Float tensor with shape [output_size, output_size].

    cut_shape:
        Shape of the retained image before resizing.

    original_shape:
        Original image shape: (height, width).

    location:
        Coordinates of the retained image in the original image:
        [row_start, row_end, col_start, col_end].
    """

   
    with Image.open(image_path) as image:
        image = image.convert("L")
        image_array = np.asarray(image, dtype=np.float32)

    original_shape = image_array.shape

    if image_array.ndim != 2:
        raise ValueError(
            f"Expected a 2-D grayscale image, got {image_array.shape} "
            f"for {image_path.name}"
        )

    if remove_black_edges:
        # Detect rows and columns that contain meaningful ultrasound content.
        #
        # A small threshold is used instead of requiring pixels to be exactly
        # zero because ultrasound borders may contain compression noise.
        foreground_threshold = 5.0

        valid_rows = np.where(
            np.mean(image_array, axis=1) > foreground_threshold
        )[0]

        valid_cols = np.where(
            np.mean(image_array, axis=0) > foreground_threshold
        )[0]

        if len(valid_rows) > 0 and len(valid_cols) > 0:
            row_start = int(valid_rows[0])
            row_end = int(valid_rows[-1]) + 1

            col_start = int(valid_cols[0])
            col_end = int(valid_cols[-1]) + 1
        else:
            # Fall back to the complete image if no foreground is found.
            row_start = 0
            row_end = original_shape[0]
            col_start = 0
            col_end = original_shape[1]

    else:
        row_start = 0
        row_end = original_shape[0]
        col_start = 0
        col_end = original_shape[1]

    cropped_image = image_array[
        row_start:row_end,
        col_start:col_end,
    ]

    if cropped_image.size == 0:
        raise ValueError(
            f"Black-edge removal produced an empty image for "
            f"{image_path.name}"
        )

    cut_shape = cropped_image.shape
    location = [row_start, row_end, col_start, col_end]

    # Resize to the stage-1 
    processed_array = resize(
        cropped_image,
        (outputsize, outputsize),
        order=3,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    # Match the common neural-network image range.
    if processed_array.max() > 1.0:
        processed_array /= 255.0

    processed_tensor = torch.from_numpy(processed_array)

    return processed_tensor, cut_shape, original_shape, location

In [80]:
# Metrics 

def get_iou(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate intersection over union for two binary arrays --> corresponds to ."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    union = np.logical_or(prediction, ground_truth).sum()

    if union == 0:
        return 1.0

    return float(intersection / union)


def get_dsc(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate Dice similarity coefficient for two binary arrays."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    denominator = prediction.sum() + ground_truth.sum()

    if denominator == 0:
        return 1.0

    return float((2.0 * intersection) / denominator)


def largest_connected_component(binary_mask: np.ndarray) -> np.ndarray:
    """Retain only the largest foreground connected component."""
    binary_mask = binary_mask.astype(bool)

    if binary_mask.sum() == 0:
        return binary_mask.astype(np.float32)

    labeled_img, num_components = sklabel(
        binary_mask,
        connectivity=1,
        background=0,
        return_num=True,
    )

    if num_components == 1:
        return binary_mask.astype(np.float32)

    component_sizes = [
        np.sum(labeled_img == component_label)
        for component_label in range(1, num_components + 1)
    ]

    largest_label = int(np.argmax(component_sizes)) + 1
    return (labeled_img == largest_label).astype(np.float32)


def calculate_stage2_roi(
    stage1_mask: np.ndarray,
    c1_size: int = 256,
) -> Tuple[int, int, int, int]:
    """
    Calculate the expanded square ROI used as input to Stage 2.

    Returns
    -------
    row_min, row_max, col_min, col_max
    """
    if stage1_mask.sum() == 0:
        min_row, min_col, max_row, max_col = 0, 0, c1_size, c1_size
    else:
        region = regionprops(stage1_mask.astype(np.uint8))[0]
        min_row, min_col, max_row, max_col = region.bbox

    row_center = (max_row + min_row) // 2
    col_center = (max_col + min_col) // 2
    max_length = max(max_row - min_row, max_col - min_col)

    large_roi_threshold = int((c1_size / 256) * 80)
    large_roi_margin = int((c1_size / 256) * 19)
    small_roi_margin = int((c1_size / 256) * 31)

    if max_length > large_roi_threshold:
        expansion = large_roi_margin + max_length // 2
    else:
        expansion = small_roi_margin + max_length // 2

    row_min = max(0, row_center - expansion)
    row_max = min(c1_size, row_center + expansion)
    col_min = max(0, col_center - expansion)
    col_max = min(c1_size, col_center + expansion)

    # IN case the crop is empty    
    if row_max <= row_min or col_max <= col_min:
        return 0, c1_size, 0, c1_size

    return row_min, row_max, col_min, col_max


# save files in order 

def discover_images(image_dir: Path) -> List[Path]:
    """Return every supported image file recursively, in stable order."""
    files = [
        path
        for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(files, key=lambda path: str(path).lower())


def build_mask_index(mask_dir: Path) -> Dict[str, Path]:
    """
    Index masks by filename stem, case-insensitively.

    The image and mask may have different filename extensions, but their stems
    must match
    """
    index: Dict[str, Path] = {}

    for path in discover_images(mask_dir):
        key = path.stem.lower()

        if key in index:
            raise ValueError(
                f"Duplicate mask stem '{path.stem}' found:\n"
                f"  {index[key]}\n"
                f"  {path}"
            )

        index[key] = path

    return index


def load_binary_mask(mask_path: Path, expected_shape: Tuple[int, int]) -> np.ndarray:
    """Read a mask as grayscale and convert it to a binary NumPy array."""
    mask = Image.open(mask_path).convert("L")
    mask_array = np.asarray(mask, dtype=np.float32)

    if mask_array.shape != expected_shape:
        raise ValueError(
            f"Ground-truth mask shape {mask_array.shape} does not match "
            f"original image shape {expected_shape} for {mask_path.name}."
        )

    return (mask_array > 0).astype(np.float32)


def save_binary_mask(mask: np.ndarray, output_path: Path) -> None:
    """Save a binary mask as an 8-bit PNG."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mask_uint8 = (mask.astype(bool).astype(np.uint8) * 255)
    Image.fromarray(mask_uint8, mode="L").save(output_path)


def save_overlay(
    image_path: Path,
    prediction: np.ndarray,
    ground_truth: Optional[np.ndarray],
    output_path: Path,
) -> None:
    """
    Save an RGB overlay:
    - red: the prediction
    - green: the ground truth (from given mask)
    - yellow: overlap
    """
    original = Image.open(image_path).convert("L")
    base = np.asarray(original, dtype=np.float32)

    if base.max() > base.min():
        base = (base - base.min()) / (base.max() - base.min())
    else:
        base = np.zeros_like(base)

    rgb = np.stack([base, base, base], axis=-1)
    prediction_bool = prediction.astype(bool)

    rgb[prediction_bool, 0] = 1.0
    rgb[prediction_bool, 1] *= 0.35
    rgb[prediction_bool, 2] *= 0.35

    if ground_truth is not None:
        ground_truth_bool = ground_truth.astype(bool)
        rgb[ground_truth_bool, 1] = 1.0
        rgb[ground_truth_bool, 0] *= 0.35
        rgb[ground_truth_bool, 2] *= 0.35

        overlap = np.logical_and(prediction_bool, ground_truth_bool)
        rgb[overlap, 0] = 1.0
        rgb[overlap, 1] = 1.0
        rgb[overlap, 2] = 0.0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8), mode="RGB").save(
        output_path
    )


# ============================================================================
# Model loading and inference
# ============================================================================

def choose_device() -> torch.device:
    """Prefer Apple MPS, then CUDA, then CPU."""
    if torch.backends.mps.is_available():
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


def extract_state_dict(checkpoint):
    """
    Support either a direct state_dict or a checkpoint dictionary containing
    a state_dict/model_state_dict field.
    """
    if not isinstance(checkpoint, dict):
        return checkpoint

    if "state_dict" in checkpoint:
        return checkpoint["state_dict"]

    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]

    return checkpoint


def load_model(
    weight_path: Path,
    device: torch.device,
    transforms,
    use_tta: bool,
) -> torch.nn.Module:
    """Create a DeepLabV3+ model and load pretrained weights."""
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")

    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b6",
        encoder_weights=None,
        in_channels=1,
        classes=1,
    )

    # CPU deserialization is typically the safest for old checkpoints.
    checkpoint = torch.load(weight_path, map_location="cpu")
    state_dict = extract_state_dict(checkpoint)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        print(
            f"Strict loading failed for {weight_path.name}; "
            "retrying with strict=False."
        )
        print(exc)
        incompatible = model.load_state_dict(state_dict, strict=False)
        print("Missing keys:", incompatible.missing_keys)
        print("Unexpected keys:", incompatible.unexpected_keys)

    model = model.to(device)
    model.eval()

    if use_tta:
        model = tta.SegmentationTTAWrapper(
            model,
            transforms,
            merge_mode="mean",
        )
        model.eval()

    return model


def run_single_image(
    image_path: Path,
    model_cascade1: torch.nn.Module,
    model_cascade2: torch.nn.Module,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run both cascades on one image.

    Returns
    -------
    final_mask:
        Binary mask restored to the original image dimensions.
    stage1_mask:
        Binary 256x256 Stage-1 largest-component mask.
    """
    processed_img, cut_shape, original_shape, location = thyroidxl_preprocess(
    image_path,
    outputsize=C1_SIZE,
    remove_black_edges=not ORIMG,
    )
    # Convert to a tensor if preprocessing returned a NumPy array.
    if processed_img.ndim != 2:
        raise ValueError(
            f"Expected processed image shape [H, W], got "
            f"{tuple(processed_img.shape)} for {image_path.name}"
        )

    image_tensor = processed_img.unsqueeze(0).unsqueeze(0)
    image_tensor = image_tensor.to(
        device=device,
        dtype=torch.float32,
    )

    assert image_tensor.shape == (1, 1, C1_SIZE, C1_SIZE), (
        f"Incorrect Stage-1 input shape: {tuple(image_tensor.shape)}"
    )

    image_array_256 = processed_img.cpu().numpy()

    with torch.inference_mode():
        stage1_logits = model_cascade1(image_tensor)
        stage1_probability = torch.sigmoid(stage1_logits)

    stage1_mask = (
        stage1_probability.squeeze().detach().cpu().numpy() > C1_THRESHOLD
    ).astype(np.float32)

    stage1_mask = largest_connected_component(stage1_mask)

    working_mask_256 = stage1_mask.copy()

    if USE_C2:
        row_min, row_max, col_min, col_max = calculate_stage2_roi(
            stage1_mask,
            C1_SIZE,
        )

        roi = image_array_256[row_min:row_max, col_min:col_max]
        roi_original_shape = roi.shape

        if roi.size == 0:
            raise RuntimeError(
                f"Stage-2 ROI is empty for {image_path.name}: "
                f"{(row_min, row_max, col_min, col_max)}"
            )

        roi_512 = resize(
            roi,
            (C2_SIZE, C2_SIZE),
            order=3,
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)

        roi_tensor = torch.from_numpy(roi_512).unsqueeze(0).unsqueeze(0)
        roi_tensor = roi_tensor.to(device=device, dtype=torch.float32)

        with torch.inference_mode():
            stage2_logits = model_cascade2(roi_tensor)
            stage2_probability = torch.sigmoid(stage2_logits)

        stage2_mask_512 = (
            stage2_probability.squeeze().detach().cpu().numpy() > C2_THRESHOLD
        ).astype(np.float32)

        stage2_mask_roi = resize(
            stage2_mask_512,
            roi_original_shape,
            order=C2_RESIZE_ORDER,
            preserve_range=True,
            anti_aliasing=False,
        )

        stage2_mask_roi = (stage2_mask_roi > 0.5).astype(np.float32)

        # Use a blank canvas so the final 256x256 prediction contains only
        # the refined Stage-2 output, rather than leftover Stage-1 pixels.
        working_mask_256 = np.zeros((C1_SIZE, C1_SIZE), dtype=np.float32)
        working_mask_256[row_min:row_max, col_min:col_max] = stage2_mask_roi

    # Reverse the resize performed by TNSCUI_preprocess.
    restored_cut_mask = resize(
        working_mask_256,
        tuple(int(value) for value in cut_shape),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )

    restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    original_shape = tuple(int(value) for value in original_shape)
    final_mask = np.zeros(original_shape, dtype=np.float32)

    row_start, row_end, col_start, col_end = [int(value) for value in location]
    target_shape = (row_end - row_start, col_end - col_start)

    if restored_cut_mask.shape != target_shape:
        restored_cut_mask = resize(
            restored_cut_mask,
            target_shape,
            order=0,
            preserve_range=True,
            anti_aliasing=False,
        )
        restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    final_mask[row_start:row_end, col_start:col_end] = restored_cut_mask
    final_mask = (final_mask > 0.5).astype(np.float32)

    return final_mask, stage1_mask



In [ ]:
def main() -> None:

    for required_path in (
        IMG_DIR,
        MASK_DIR,
    ):

        if not required_path.exists():

            raise FileNotFoundError(
                f"Directory not found: "
                f"{required_path}"
            )

    if not SCREENED_TXT.exists():

        raise FileNotFoundError(
            f"Screened file not found: "
            f"{SCREENED_TXT}"
        )


    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    PREDICTION_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    if SAVE_OVERLAYS:

        OVERLAY_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        PATIENT_COMPARISON_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )


    # Device


    device = choose_device()

    print(
        f"Using device: {device}"
    )

 
    # Load SCREENED patients


    patients = load_screened_patients(
        SCREENED_TXT
    )

    print(
        f"Screened patients: "
        f"{len(patients)}"
    )

    total_screened_images = sum(
        len(images)
        for images
        in patients.values()
    )

    print(
        f"Screened images: "
        f"{total_screened_images}"
    )


    # Build image + mask indexes


    image_index_map = build_image_index(
        IMG_DIR
    )

    mask_index = build_mask_index(
        MASK_DIR
    )

    print(
        f"Raw images indexed: "
        f"{len(image_index_map)}"
    )

    print(
        f"Masks indexed: "
        f"{len(mask_index)}"
    )


    # TTA


    tta_transforms = tta.Compose(
        [
            tta.VerticalFlip(),
            tta.HorizontalFlip(),
            tta.Rotate90(
                angles=[
                    0,
                    180,
                ]
            ),
        ]
    )

    # ==========================================================    # Models


    print(
        "Loading Stage-1 model..."
    )

    model_cascade1 = load_model(
        WEIGHT_C1,
        device,
        tta_transforms,
        C1_TTA,
    )

    print(
        "Loading Stage-2 model..."
    )

    model_cascade2 = load_model(
        WEIGHT_C2,
        device,
        tta_transforms,
        C2_TTA,
    )


    # Results

    results: List[Dict] = []

    failed_count = 0

    # ==========================================================
    # OUTER LOOP = PATIENT
    # ==========================================================

    for patient_number, (
        patient_id,
        patient_images,
    ) in enumerate(
        sorted(patients.items()),
        start=1,
    ):

        print(
            "\n"
            + "=" * 72
        )

        print(
            f"PATIENT "
            f"[{patient_number}/"
            f"{len(patients)}] "
            f"{patient_id}"
        )

        print(
            f"Screened images: "
            f"{len(patient_images)}"
        )

        print(
            "=" * 72
        )

        patient_results = []


        # INNER LOOP = images/views belonging to patient
       
        for image_number, image_info in enumerate(
            patient_images,
            start=1,
        ):

            image_id = (
                image_info[
                    "image_id"
                ]
                .strip()
            )

            view = (
                image_info[
                    "view"
                ]
                .strip()
            )

            image_key = (
                image_id.lower()
            )

            print(
                f"\n  "
                f"[{image_number}/"
                f"{len(patient_images)}] "
                f"{image_id}"
            )

            print(
                f"  View: {view}"
            )

            # --------------------------------------------------
            # Find image
            # --------------------------------------------------

            if (
                image_key
                not in image_index_map
            ):

                print(
                    "  ERROR: image not found."
                )

                failed_count += 1

                result = {
                    "patient_id":
                        patient_id,

                    "image_id":
                        image_id,

                    "view":
                        view,

                    "image_name":
                        "",

                    "image_path":
                        "",

                    "mask_path":
                        "",

                    "prediction_path":
                        "",

                    "overlay_path":
                        "",

                    "height":
                        "",

                    "width":
                        "",

                    "stage1_foreground_pixels":
                        "",

                    "final_foreground_pixels":
                        "",

                    "ground_truth_foreground_pixels":
                        "",

                    "iou":
                        "",

                    "dsc":
                        "",

                    "iou_below_0_3":
                        "",

                    "status":
                        "failed",

                    "error":
                        "Image not found",
                }

                results.append(result)
                patient_results.append(
                    result
                )

                continue

            image_path = (
                image_index_map[
                    image_key
                ]
            )

            # --------------------------------------------------
            # Find the mask of the image 
     

            if image_key not in mask_index:

                print(
                    "  ERROR: mask not found."
                )

                failed_count += 1

                result = {
                    "patient_id":
                        patient_id,

                    "image_id":
                        image_id,

                    "view":
                        view,

                    "image_name":
                        image_path.name,

                    "image_path":
                        str(image_path),

                    "mask_path":
                        "",

                    "prediction_path":
                        "",

                    "overlay_path":
                        "",

                    "height":
                        "",

                    "width":
                        "",

                    "stage1_foreground_pixels":
                        "",

                    "final_foreground_pixels":
                        "",

                    "ground_truth_foreground_pixels":
                        "",

                    "iou":
                        "",

                    "dsc":
                        "",

                    "iou_below_0_3":
                        "",

                    "status":
                        "failed",

                    "error":
                        "Mask not found",
                }

                results.append(result)
                patient_results.append(
                    result
                )

                continue

            mask_path = (
                mask_index[
                    image_key
                ]
            )

            try:

                # Original dimensions


                with Image.open(
                    image_path
                ) as original_image:

                    original_shape = (
                        original_image.height,
                        original_image.width,
                    )

                # Ground truth
  

                ground_truth = (
                    load_binary_mask(
                        mask_path,
                        original_shape,
                    )
                )

                # ==============================================
                # Full TNSCUI inference
                #
                # thyroidxl_preprocess() is called INSIDE
                # run_single_image().
                # ==============================================

                final_mask, stage1_mask = (
                    run_single_image(
                        image_path,
                        model_cascade1,
                        model_cascade2,
                        device,
                    )
                )

                if (
                    final_mask.shape
                    != ground_truth.shape
                ):

                    raise ValueError(
                        f"Prediction shape "
                        f"{final_mask.shape} "
                        "does not match "
                        f"GT shape "
                        f"{ground_truth.shape}"
                    )

                # ==============================================
                # Metrics
                # ==============================================

                iou = get_iou(
                    final_mask,
                    ground_truth,
                )

                dsc = get_dsc(
                    final_mask,
                    ground_truth,
                )

                print(
                    f"  IoU: {iou:.4f}"
                )

                print(
                    f"  DSC: {dsc:.4f}"
                )

                # ==============================================
                # Prediction
                # ==============================================

                prediction_path = (
                    PREDICTION_DIR
                    / f"{image_id}.png"
                )

                save_binary_mask(
                    final_mask,
                    prediction_path,
                )

                # ==============================================
                # Overlay
                # ==============================================

                if SAVE_OVERLAYS:

                    patient_overlay_dir = (
                        OVERLAY_DIR
                        / patient_id
                    )

                    overlay_path = (
                        patient_overlay_dir
                        / (
                            f"{image_id}_"
                            f"{view}_overlay.png"
                        )
                    )

                    save_overlay(
                        image_path,
                        final_mask,
                        ground_truth,
                        overlay_path,
                    )

                else:

                    overlay_path = None

                # ==============================================
                # Result row
                # ==============================================

                result = {
                    "patient_id":
                        patient_id,

                    "image_id":
                        image_id,

                    "view":
                        view,

                    "image_name":
                        image_path.name,

                    "image_path":
                        str(image_path),

                    "mask_path":
                        str(mask_path),

                    "prediction_path":
                        str(
                            prediction_path
                        ),

                    "overlay_path":
                        (
                            str(overlay_path)
                            if overlay_path
                            else ""
                        ),

                    "height":
                        original_shape[0],

                    "width":
                        original_shape[1],

                    "stage1_foreground_pixels":
                        int(
                            stage1_mask.sum()
                        ),

                    "final_foreground_pixels":
                        int(
                            final_mask.sum()
                        ),

                    "ground_truth_foreground_pixels":
                        int(
                            ground_truth.sum()
                        ),

                    "iou":
                        iou,

                    "dsc":
                        dsc,

                    "iou_below_0_3":
                        int(
                            iou < 0.3
                        ),

                    "status":
                        "ok",

                    "error":
                        "",
                }

                results.append(result)

                patient_results.append(
                    result
                )

            except Exception as exc:

                failed_count += 1

                print(
                    f"ERROR processing "
                    f"{image_id}: {exc}"
                )

                traceback.print_exc()

                result = {
                    "patient_id":
                        patient_id,

                    "image_id":
                        image_id,

                    "view":
                        view,

                    "image_name":
                        image_path.name,

                    "image_path":
                        str(image_path),

                    "mask_path":
                        str(mask_path),

                    "prediction_path":
                        "",

                    "overlay_path":
                        "",

                    "height":
                        "",

                    "width":
                        "",

                    "stage1_foreground_pixels":
                        "",

                    "final_foreground_pixels":
                        "",

                    "ground_truth_foreground_pixels":
                        "",

                    "iou":
                        "",

                    "dsc":
                        "",

                    "iou_below_0_3":
                        "",

                    "status":
                        "failed",

                    "error":
                        str(exc),
                }

                results.append(result)

                patient_results.append(
                    result
                )

                if not CONTINUE_ON_ERROR:
                    raise


        # Patient finished
     

        successful_patient_results = [
            row
            for row in patient_results
            if row["status"] == "ok"
        ]

        # ------------------------------------------------------
        # Side-by-side overlay
        # ------------------------------------------------------

        if (
            SAVE_OVERLAYS
            and len(
                successful_patient_results
            ) >= 2
        ):

            comparison_path = (
                PATIENT_COMPARISON_DIR
                / (
                    f"{patient_id}"
                    f"_comparison.png"
                )
            )

            save_patient_comparison(
                patient_id,
                successful_patient_results,
                comparison_path,
            )

            print(
                f"\n  Comparison saved:"
                f"\n  {comparison_path}"
            )


        # Patient running metrics


        if successful_patient_results:

            patient_ious = np.asarray(
                [
                    row["iou"]
                    for row
                    in successful_patient_results
                ],
                dtype=np.float64,
            )

            patient_dscs = np.asarray(
                [
                    row["dsc"]
                    for row
                    in successful_patient_results
                ],
                dtype=np.float64,
            )

            print(
                f"\n  Patient mean IoU: "
                f"{patient_ious.mean():.4f}"
            )

            print(
                f"  Patient mean DSC: "
                f"{patient_dscs.mean():.4f}"
            )

    # ==========================================================
    # IMAGE-LEVEL METRICS CSV
    # ==========================================================

    fieldnames = [
        "patient_id",
        "image_id",
        "view",

        "image_name",
        "image_path",
        "mask_path",

        "prediction_path",
        "overlay_path",

        "height",
        "width",

        "stage1_foreground_pixels",
        "final_foreground_pixels",
        "ground_truth_foreground_pixels",

        "iou",
        "dsc",
        "iou_below_0_3",

        "status",
        "error",
    ]

    with METRICS_CSV.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(
            results
        )


    # PATIENT-LEVEL METRICS CSV


    save_patient_metrics(
        results,
        PATIENT_METRICS_CSV,
    )

    # ==========================================================
    # Dataset summary
    # ==========================================================

    successful_results = [
        row
        for row in results
        if row["status"] == "ok"
    ]

    print(
        "\n"
        + "=" * 72
    )

    print(
        "SCREENED PATIENT INFERENCE COMPLETE"
    )

    print(
        "=" * 72
    )

    print(
        f"Patients screened: "
        f"{len(patients)}"
    )

    print(
        f"Images screened: "
        f"{total_screened_images}"
    )

    print(
        f"Successfully processed: "
        f"{len(successful_results)}"
    )

    print(
        f"Failed: "
        f"{failed_count}"
    )

  
    # Overall image-level metrics

    if successful_results:

        ious = np.asarray(
            [
                row["iou"]
                for row
                in successful_results
            ],
            dtype=np.float64,
        )

        dscs = np.asarray(
            [
                row["dsc"]
                for row
                in successful_results
            ],
            dtype=np.float64,
        )

        print(
            f"Mean IoU: "
            f"{ious.mean():.4f}"
        )

        print(
            f"Median IoU: "
            f"{np.median(ious):.4f}"
        )

        print(
            f"Mean DSC: "
            f"{dscs.mean():.4f}"
        )

        print(
            f"Median DSC: "
            f"{np.median(dscs):.4f}"
        )

        print(
            f"IoU below 0.3: "
            f"{int(np.sum(ious < 0.3))}"
        )

    print(
        f"\nImage metrics CSV: "
        f"{METRICS_CSV}"
    )

    print(
        f"Patient metrics CSV: "
        f"{PATIENT_METRICS_CSV}"
    )

    print(
        f"Predicted masks: "
        f"{PREDICTION_DIR}"
    )

    if SAVE_OVERLAYS:

        print(
            f"Individual overlays: "
            f"{OVERLAY_DIR}"
        )

        print(
            f"Patient comparisons: "
            f"{PATIENT_COMPARISON_DIR}"
        )


if __name__ == "__main__":
    main()

Using device: mps
Screened patients: 839
Screened images: 949
Raw images indexed: 9541
Masks indexed: 9541
Loading Stage-1 model...
Loading Stage-2 model...

PATIENT [1/839] 00000127
Screened images: 1

  [1/1] 00000127_1914A778_1
  View: Unknown
  IoU: 0.7176
  DSC: 0.8356

  Patient mean IoU: 0.7176
  Patient mean DSC: 0.8356

PATIENT [2/839] 00000139
Screened images: 2

  [1/2] 00000139_FA7E0544_0
  View: Longitudinal


/var/folders/gd/p6nf_1w11938t89t0bj2r9lc0000gn/T/ipykernel_42433/1610858530.py:153: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask_uint8, mode="L").save(output_path)
/var/folders/gd/p6nf_1w11938t89t0bj2r9lc0000gn/T/ipykernel_42433/1610858530.py:195: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8), mode="RGB").save(


  IoU: 0.4744
  DSC: 0.6435

  [2/2] 00000139_217557BB_1
  View: Unknown
  IoU: 0.8230
  DSC: 0.9029

  Comparison saved:
  /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/inference_outputs_long_vs_trans_v1_07/patient_view_comparisons/00000139_comparison.png

  Patient mean IoU: 0.6487
  Patient mean DSC: 0.7732

PATIENT [3/839] 00000141
Screened images: 1

  [1/1] 00000141_CFD4B94F_0
  View: Transverse
  IoU: 0.9106
  DSC: 0.9532

  Patient mean IoU: 0.9106
  Patient mean DSC: 0.9532

PATIENT [4/839] 00000146
Screened images: 1

  [1/1] 00000146_8EC61D31_1
  View: Transverse
  IoU: 0.8351
  DSC: 0.9102

  Patient mean IoU: 0.8351
  Patient mean DSC: 0.9102

PATIENT [5/839] 00000152
Screened images: 1

  [1/1] 00000152_FED1C9E5_2
  View: Transverse
  IoU: 0.7855
  DSC: 0.8799

  Patient mean IoU: 0.7855
  Patient mean DSC: 0.8799

PATIENT [6/839] 00000156
Screened images: 1

  [1/1] 00000156_D1280CBD_2
  View: Longitudinal
  IoU: 0.6420
  DSC: 0.7819


# ANALYSIS OF PATIENT METRICS

In [84]:
patient_metrics = pd.read_csv(PATIENT_METRICS_CSV)

In [85]:
patient_metrics

,patient_id,total_images,transverse_images,longitudinal_images,unknown_images,transverse_mean_iou,longitudinal_mean_iou,iou_long_minus_trans,transverse_mean_dsc,longitudinal_mean_dsc,dsc_long_minus_trans,has_both_views
0,127,1,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,0
1,139,2,0,1,1,NaN,0.474364,NaN,NaN,0.643483,NaN,0
2,141,1,1,0,0,0.910585,NaN,NaN,0.953200,NaN,NaN,0
3,146,1,1,0,0,0.835139,NaN,NaN,0.910164,NaN,NaN,0
4,152,1,1,0,0,0.785524,NaN,NaN,0.879881,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
834,4739,1,1,0,0,0.912463,NaN,NaN,0.954228,NaN,NaN,0
835,4751,1,1,0,0,0.892597,NaN,NaN,0.943251,NaN,NaN,0
836,4756,1,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,0
837,4762,1,1,0,0,0.795966,NaN,NaN,0.886393,NaN,NaN,0


In [88]:
patient_metrics['has_both_views'].value_counts()

has_both_views
0    794
1     45
Name: count, dtype: int64

In [89]:
two_views = patient_metrics[patient_metrics['has_both_views'] == True]

In [91]:
two_views[[
    "patient_id",
    "transverse_mean_iou",
    "longitudinal_mean_iou",
    "iou_long_minus_trans",
    "transverse_mean_dsc",
    "longitudinal_mean_dsc",
    "dsc_long_minus_trans"
]]

,patient_id,transverse_mean_iou,longitudinal_mean_iou,iou_long_minus_trans,transverse_mean_dsc,longitudinal_mean_dsc,dsc_long_minus_trans
6,187,0.743176,0.843781,0.100605,0.852669,0.915272,0.062604
20,248,0.844813,0.884262,0.039448,0.915879,0.938576,0.022697
32,333,0.842122,0.767007,-0.075115,0.914296,0.868143,-0.046153
38,372,0.833796,0.863882,0.030086,0.909366,0.926971,0.017604
48,441,0.687611,0.694949,0.007338,0.814893,0.820024,0.005131
124,863,0.696815,0.759077,0.062262,0.821321,0.863040,0.041719
135,968,0.460851,0.637814,0.176964,0.630935,0.778860,0.147926
150,1071,0.564799,0.661403,0.096605,0.721880,0.796198,0.074318
176,1175,0.722246,0.679650,-0.042596,0.838726,0.809276,-0.029450
186,1223,0.847556,0.765992,-0.081564,0.917489,0.867383,-0.050105


In [92]:
print("Number of paired patients:", len(two_views))

print("\nIoU")
print("Transverse mean:", two_views["transverse_mean_iou"].mean())
print("Longitudinal mean:", two_views["longitudinal_mean_iou"].mean())
print("Mean difference:",
      two_views["iou_long_minus_trans"].mean())

print("\nDSC")
print("Transverse mean:", two_views["transverse_mean_dsc"].mean())
print("Longitudinal mean:", two_views["longitudinal_mean_dsc"].mean())
print("Mean difference:",
      two_views["dsc_long_minus_trans"].mean())

Number of paired patients: 45

IoU
Transverse mean: 0.7250897034303594
Longitudinal mean: 0.723187074750881
Mean difference: -0.0019026286794786634

DSC
Transverse mean: 0.8250129154514214
Longitudinal mean: 0.8218437990392091
Mean difference: -0.003169116412212182


In [93]:
two_views = two_views.dropna(subset=[
    "transverse_mean_iou",
    "longitudinal_mean_iou",
    "transverse_mean_dsc",
    "longitudinal_mean_dsc"
])

In [95]:
# IoU
iou_long_higher = (two_views["iou_long_minus_trans"] > 0).sum()
iou_trans_higher = (two_views["iou_long_minus_trans"] < 0).sum()
iou_equal = (two_views["iou_long_minus_trans"] == 0).sum()

# DSC
dsc_long_higher = (two_views["dsc_long_minus_trans"] > 0).sum()
dsc_trans_higher = (two_views["dsc_long_minus_trans"] < 0).sum()
dsc_equal = (two_views["dsc_long_minus_trans"] == 0).sum()

print(f"Total paired patients: {len(two_views)}")

print("\nIoU:")
print(f"Longitudinal higher: {iou_long_higher}")
print(f"Transverse higher:   {iou_trans_higher}")
print(f"Equal:               {iou_equal}")

print("\nDSC:")
print(f"Longitudinal higher: {dsc_long_higher}")
print(f"Transverse higher:   {dsc_trans_higher}")
print(f"Equal:               {dsc_equal}")

Total paired patients: 45

IoU:
Longitudinal higher: 27
Transverse higher:   17
Equal:               1

DSC:
Longitudinal higher: 27
Transverse higher:   17
Equal:               1
